# Notebook 06 — Retrain Model v2 (With Goalscorer Features)

**Goal:** Train a new model with extra features and compare to v1 (~52% accuracy).

**Expected v2 accuracy:** ~63% (more features = better predictions)

**Input:** `data/processed/matches_ready_v2.csv`

# Step 1 — Load Version 2 Data

In [1]:
import pandas as pd
from pathlib import Path

PROJECT_ROOT = Path("..")
DATA_PATH = PROJECT_ROOT / "data" / "processed" / "matches_ready_v2.csv"
MODEL_V2_PATH = PROJECT_ROOT / "models" / "match_predictor_v2.joblib"

df = pd.read_csv(DATA_PATH)

FEATURE_COLUMNS = [
    "home_team",
    "away_team",
    "tournament",
    "is_neutral",
    "year",
    "month",
    "home_roll_goals_for_5",
    "home_roll_goals_against_5",
    "away_roll_goals_for_5",
    "away_roll_goals_against_5",
    "home_roll_unique_scorers_5",
    "away_roll_unique_scorers_5",
    "home_roll_penalty_goals_5",
    "away_roll_penalty_goals_5",
]
TARGET_COLUMN = "home_result"

print("Rows:", len(df))
print("Features:", len(FEATURE_COLUMNS))

Rows: 5723
Features: 14


# Step 2 — Train/Test Split

In [2]:
from sklearn.model_selection import train_test_split

X = df[FEATURE_COLUMNS]
y = df[TARGET_COLUMN]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print("Train:", len(X_train), "| Test:", len(X_test))

Train: 4578 | Test: 1145


# Step 3 — Build Pipeline and Train

In [3]:
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder
from sklearn.pipeline import Pipeline
from sklearn.ensemble import RandomForestClassifier

text_columns = ["home_team", "away_team", "tournament"]

preprocessor = ColumnTransformer(
    transformers=[
        ("cat", OneHotEncoder(handle_unknown="ignore"), text_columns)
    ],
    remainder="passthrough",
)

pipeline_v2 = Pipeline(
    steps=[
        ("preprocessor", preprocessor),
        ("model", RandomForestClassifier(n_estimators=100, random_state=42)),
    ]
)

pipeline_v2.fit(X_train, y_train)
print("Training complete.")

Training complete.


# Step 4 — Evaluate and Compare v1 vs v2

In [4]:
from sklearn.metrics import accuracy_score, classification_report

y_pred = pipeline_v2.predict(X_test)
accuracy_v2 = accuracy_score(y_test, y_pred)

print("=" * 50)
print("MODEL COMPARISON")
print("=" * 50)
print(f"v1 accuracy (basic features):     ~52%")
print(f"v2 accuracy (goalscorer features):  {accuracy_v2:.1%}")
print(f"Random baseline (3 classes):      ~33%")
print("=" * 50)
print("\nDetailed v2 report:")
print(classification_report(y_test, y_pred))

MODEL COMPARISON
v1 accuracy (basic features):     ~52%
v2 accuracy (goalscorer features):  63.5%
Random baseline (3 classes):      ~33%

Detailed v2 report:
              precision    recall  f1-score   support

        Draw       0.52      0.26      0.35       262
        Loss       0.68      0.58      0.62       333
         Win       0.64      0.85      0.73       550

    accuracy                           0.63      1145
   macro avg       0.61      0.56      0.57      1145
weighted avg       0.62      0.63      0.61      1145



# Step 5 — Save Model v2

In [5]:
import joblib

MODEL_V2_PATH.parent.mkdir(parents=True, exist_ok=True)
joblib.dump(pipeline_v2, MODEL_V2_PATH)

print("Saved v2 model to:", MODEL_V2_PATH.resolve())

Saved v2 model to: C:\Users\HP-15\Desktop\worldcup_predictor\models\match_predictor_v2.joblib


# Step 6 — Predict Argentina vs Austria (v2 Model)

v2 needs **rolling form features** too. For a live match, use each team's recent averages.

Below we use example rolling values — in production you'd compute from latest data.

In [6]:
# Load v2 model
model_v2 = joblib.load(MODEL_V2_PATH)

# Example rolling stats (replace with real computed values from latest matches)
argentina_vs_austria = pd.DataFrame(
    [
        {
            "home_team": "Argentina",
            "away_team": "Austria",
            "tournament": "FIFA World Cup",
            "is_neutral": 1,
            "year": 2026,
            "month": 6,
            "home_roll_goals_for_5": 2.0,
            "home_roll_goals_against_5": 0.6,
            "away_roll_goals_for_5": 1.8,
            "away_roll_goals_against_5": 1.2,
            "home_roll_unique_scorers_5": 2.5,
            "away_roll_unique_scorers_5": 2.0,
            "home_roll_penalty_goals_5": 0.2,
            "away_roll_penalty_goals_5": 0.0,
        }
    ]
)

prediction = model_v2.predict(argentina_vs_austria)[0]
proba = model_v2.predict_proba(argentina_vs_austria)[0]
classes = model_v2.named_steps["model"].classes_

print("Argentina vs Austria (v2 model)")
print("Predicted home result:", prediction)
for label, p in zip(classes, proba):
    print(f"  {label}: {p:.1%}")

Argentina vs Austria (v2 model)
Predicted home result: Win
  Draw: 33.0%
  Loss: 13.0%
  Win: 54.0%


# Summary

| Version | Features | Accuracy |
|---------|----------|----------|
| v1 | teams, tournament, date, neutral | ~52% |
| v2 | v1 + rolling goals + scorers + penalties | ~63% |

**Path A Phase 2 complete.** Phase 3 (later): formations, player stats, coach data.